Testing for importing the datasets and making sure the script works properly in and of itself before integration.

Test the datasets are loading properly

In [13]:
import sys
from pathlib import Path
import pe_dataset

MAX_BYTES = int(1.024 * 10**6)
malicious_path = Path("../Data/Testing_set/malicious")
benign_path = Path("../Data/Testing_set/benign")

data = pe_dataset.build_dataset_from_dir(benign_dir=benign_path, malware_dir=malicious_path, max_bytes=MAX_BYTES)


Getting some output of the dataset to see what the tensors are returning

In [14]:
import random

i = 0

while i < 200:
    index = random.randint(0, len(data))
    print(data.__getitem__(idx=index))

    i+=1

(tensor([ 77,  90, 144,  ..., 256, 256, 256]), tensor(0))
(tensor([ 77,  90, 144,  ..., 256, 256, 256]), tensor(0))
(tensor([ 77,  90, 144,  ..., 256, 256, 256]), tensor(0))
(tensor([120, 156, 236,  ..., 256, 256, 256]), tensor(1))
(tensor([120, 156, 236,  ..., 256, 256, 256]), tensor(1))
(tensor([120, 156, 204,  ..., 256, 256, 256]), tensor(1))
(tensor([120, 156, 236,  ..., 256, 256, 256]), tensor(1))
(tensor([120, 156, 236,  ..., 256, 256, 256]), tensor(1))
(tensor([120, 156, 237,  ..., 256, 256, 256]), tensor(1))
(tensor([ 77,  90, 144,  ..., 256, 256, 256]), tensor(0))
(tensor([ 77,  90, 144,  ..., 256, 256, 256]), tensor(0))
(tensor([ 77,  90, 144,  ..., 256, 256, 256]), tensor(0))
(tensor([120, 156, 236,  ..., 256, 256, 256]), tensor(1))
(tensor([ 77,  90, 144,  ..., 256, 256, 256]), tensor(0))
(tensor([120, 156, 236,  ..., 256, 256, 256]), tensor(1))
(tensor([120, 156, 204,  ..., 256, 256, 256]), tensor(1))
(tensor([ 77,  90, 144,  ..., 256, 256, 256]), tensor(0))
(tensor([120, 

KeyboardInterrupt: 

For the the training loop I must:

1. Train the defender normally

2. Check if the warm up phase for the defender is exceeded

3. Discover what predictions the model made on a batch of malicious files

4. For the files that were correctly classified, place those into the attacker's input source

5. Train the attacker on the passed malicious binaries and run preturbations

6. Track whether or not each perturbation succeeds

7. Repeat 

First we need to split the dataset as usual

In [15]:
import agents
import config

# This exists within pe_dataset.py, but I have it here to make things easier to test: changes made to pe_dataset.py require a reload
def split(dataset: pe_dataset.PEBinaryDataset, test_split=0.3, rand_state=42):
    from sklearn.model_selection import train_test_split

    data_labels = dataset.labels

    train_paths, test_paths, train_labels, test_labels = train_test_split(
        dataset.file_paths, data_labels, test_size=test_split,
        random_state=rand_state, stratify=data_labels)

    return train_paths, test_paths, train_labels, test_labels

NUM_EPOCHS = 20

(X_train_paths, X_test_paths, y_train, y_test) = split(data)

num_benign, num_mal = 0, 0

for label in y_train:
    if label == 1:
        num_benign+=1
    else:
        num_mal+=1

if num_mal == num_benign+1 or num_mal == num_benign-1 or num_mal == num_benign:
    print("Benign/Malicious split is as balanced as it can be (there's always going to be 1 more benign or malicious)")
else:
    print(f"Benign/Malicious balance: 0->{num_benign}, 1->{num_mal}")


Benign/Malicious split is as balanced as it can be (there's always going to be 1 more benign or malicious)


In [30]:
from torch.utils.data import DataLoader
from Models.MalConv2 import MalConv

import torch
import torch.optim as optim
import torch.nn as nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
LEARNING_RATE = 1e-3

train_data = pe_dataset.PEBinaryDataset(file_paths=X_train_paths, labels=y_train, max_bytes=MAX_BYTES)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)

test_data = pe_dataset.PEBinaryDataset(file_paths=X_test_paths, labels=y_test, max_bytes=MAX_BYTES)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=True)

model = MalConv.MalConv()
model = model.to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

_, _, train_whats_happening = train_data.filter_readable_labels()
if len(train_whats_happening) != 0:
    print("Bad paths in training set:")
    for entry in train_whats_happening:
        print(f"Bad Paths: {entry[0]} | Label: {entry[1]} | Error: {entry[2]}")

_, _, test_whats_happening = test_data.filter_readable_labels()
if len(test_whats_happening) != 0:
    print("Bad paths in test dataset")
    for entryy in test_whats_happening:
        print(f"Bad Paths: {entry[0]} | Label: {entry[1]} | Error: {entry[2]}")


In [ ]:
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()
        output = model(x)
        logits = output[0] if isinstance(output, tuple) else output

        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        preds = torch.argmax(logits, dim=1)
        total_correct += (preds == y).sum().item()
        total_samples += x.size(0)


    return total_loss / total_samples, total_correct / total_samples

@torch.no_grad()
def run_eval(model, loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        output = model(x)
        logits = output[0] if isinstance(output, tuple) else output

        loss = criterion(logits, y)

        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        total_correct += (preds == y).sum().item()
        total_samples += x.size(0)

    return total_loss / total_samples, total_correct / total_samples

best_val_loss = float("inf")
loss_prev = 99999999

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader)

    print(
        f"Epoch {epoch+1}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
    )

    if train_loss > loss_prev:
        print(f"Previous loss ({loss_prev}) is less than current epoch loss ({loss}). Stopping early to circumvent overfitting")
        break

    if train_loss <= 0.0001:
        print("Loss has effectively reached 0 - there is no point to continue training.")
        break


test_loss, test_acc = run_eval(model, test_loader)
print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

Epoch 1/20 | Train Loss: 0.0000, Train Acc: 1.0000 | 
Loss has reached 0 - there is no point to continue training.
Test Loss: 0.0834, Test Acc: 0.9667


# PROJECT NOTES
## Data
- Benign data gathered from clean windows 10 ISO mounting and extraction of Windows 10 home.
- Malicious data was gathered from SOREL-20M dataset
    - The original idea for the EMBER2024 dataset fell through, as this dataset does NOT provide the raw binaries that I planned to utilize
    - The setup script needs to be revised and reworked to accomodate the new setup methodology
        - This also means that the README is also out of date, and needs to have its setup section rewritten
    - Families of malicious data were NOT tracked, thus any imbalance of malware family representation (ie: virus, trojan, backdoor, or worm) have not been tracked and, as far as I know, are not able to be tracked
        - This should not have an effect on defender robustness improvements, but that isn't something measured by this study

- Models are restricted to MalConv and _____
    - This is intentional, at least for now
        - This is more effective than to introduce a new variable in unproven model architectures against a benchmark, rather than training the benchmark against its experminental self.